# 01 Random Forest Regressor (Macro)

Predict next-quarter real GDP growth with a Random Forest. Compare RMSE / MAE / R² against the OLS baseline from notebook `00`.

## Table of Contents
- [Why Random Forest for regression](#why)
- [Environment bootstrap](#bootstrap)
- [Load data + split](#load-split)
- [Fit a default Random Forest](#fit-default)
- [Hyperparameter search via walk-forward CV](#tune)
- [Evaluate the tuned model](#evaluate)
- [Predicted vs actual + residuals](#diagnostics)
- [Permutation importance](#importance)
- [Compare to OLS baseline](#compare)
- [Reflection](#reflection)

<a id="why"></a>
## Why Random Forest for Regression

Random Forest fits many decision trees on bootstrap samples of the data and averages their predictions. The averaging is the whole point: a single deep tree memorizes its training data (high variance, see `00a/08`); averaging many decorrelated trees keeps the bias low while collapsing the variance.

**When you would expect RF to beat OLS on a macro target:**
- The relationship is **nonlinear** (e.g., a yield-curve inversion has a step-change effect on GDP rather than a smooth linear one).
- The relationship has **interactions** (e.g., low fed funds is recessionary only when unemployment is already high).
- Outliers and recessions distort OLS coefficients, but trees are scale-free and robust.

**When OLS will win:**
- The true relationship is approximately linear (very common in macro).
- The sample is small (~150 quarters) — RF needs data to discover nonlinearities.
- You need confidence intervals on coefficients (use OLS for that, *and* RF for prediction).

<a id="bootstrap"></a>
## Environment Bootstrap

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    p = start
    for _ in range(8):
        if (p / 'src').exists() and (p / 'docs').exists():
            return p
        p = p.parent
    raise RuntimeError('Could not find repo root.')


PROJECT_ROOT = find_repo_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
SAMPLE_DIR = PROJECT_ROOT / 'data' / 'sample'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
print('Repo root:', PROJECT_ROOT)

<a id="load-split"></a>
## Load Data + Split

Identical to notebook `00`. Same target, same features, same split — so the metrics are directly comparable.

In [ ]:
import numpy as np
import pandas as pd
from src.evaluation import time_train_test_split_index

path = PROCESSED_DIR / 'macro_quarterly.csv'
if path.exists():
    df = pd.read_csv(path, index_col=0, parse_dates=True)
else:
    df = pd.read_csv(SAMPLE_DIR / 'macro_quarterly_sample.csv', index_col=0, parse_dates=True)

y_col = 'gdp_growth_qoq'
x_cols = ['T10Y2Y_lag1', 'UNRATE_lag1', 'FEDFUNDS_lag1', 'INDPRO_lag1', 'RSAFS_lag1']
data = df[[y_col] + x_cols].dropna().copy()

split = time_train_test_split_index(len(data), test_size=0.2)
train, test = data.iloc[split.train_slice], data.iloc[split.test_slice]
X_train, y_train = train[x_cols].to_numpy(), train[y_col].to_numpy()
X_test, y_test = test[x_cols].to_numpy(), test[y_col].to_numpy()
print(f'Train: {len(train)}   Test: {len(test)}')

<a id="fit-default"></a>
## Fit a Default Random Forest

Start with sensible defaults. Trees do not need feature scaling, so we feed raw values directly.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from src.evaluation import regression_metrics

rf_default = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    random_state=0,
    n_jobs=-1,
)
rf_default.fit(X_train, y_train)
yhat_default = rf_default.predict(X_test)
default_metrics = regression_metrics(y_test, yhat_default)
default_metrics

<a id="tune"></a>
## Hyperparameter Search via Walk-Forward CV

Random Forest has three knobs that matter most for small macro datasets:
- `n_estimators`: more trees ≈ better, until diminishing returns. 500 is a fine default.
- `max_depth`: shallower trees regularize harder. Try `None` (full growth) vs 3 vs 6.
- `min_samples_leaf`: bigger leaves ≈ stronger regularization. Try 1, 3, 5, 10.

We score each candidate using **walk-forward CV**, not random K-fold, because the data is a time series.

In [ ]:
from itertools import product
from src.evaluation import walk_forward_splits

X_full = data[x_cols].to_numpy()
y_full = data[y_col].to_numpy()
n = len(data)

grid = {
    'max_depth': [3, 6, None],
    'min_samples_leaf': [1, 3, 5],
}

results = []
for max_depth, msl in product(grid['max_depth'], grid['min_samples_leaf']):
    fold_rmse = []
    for sp in walk_forward_splits(n, initial_train_size=int(0.5 * n), test_size=8, step_size=8):
        m = RandomForestRegressor(n_estimators=300, max_depth=max_depth,
                                  min_samples_leaf=msl, random_state=0, n_jobs=-1)
        m.fit(X_full[sp.train_slice], y_full[sp.train_slice])
        yp = m.predict(X_full[sp.test_slice])
        fold_rmse.append(regression_metrics(y_full[sp.test_slice], yp)['rmse'])
    results.append({'max_depth': max_depth, 'min_samples_leaf': msl,
                    'cv_rmse_mean': float(np.mean(fold_rmse)),
                    'cv_rmse_std': float(np.std(fold_rmse))})

results_df = pd.DataFrame(results).sort_values('cv_rmse_mean').reset_index(drop=True)
results_df

<a id="evaluate"></a>
## Evaluate the Tuned Model

Pick the hyperparameters with the lowest CV RMSE, retrain on the full *training* set, and score on the held-out test set.

In [ ]:
best = results_df.iloc[0]
best_max_depth = None if pd.isna(best['max_depth']) else int(best['max_depth'])
best_msl = int(best['min_samples_leaf'])
print(f'Best params: max_depth={best_max_depth}, min_samples_leaf={best_msl}')

rf_tuned = RandomForestRegressor(
    n_estimators=500,
    max_depth=best_max_depth,
    min_samples_leaf=best_msl,
    random_state=0,
    n_jobs=-1,
)
rf_tuned.fit(X_train, y_train)
yhat_tuned = rf_tuned.predict(X_test)
tuned_metrics = regression_metrics(y_test, yhat_tuned)
tuned_metrics

<a id="diagnostics"></a>
## Predicted vs Actual + Residuals

Two plots that should be reflexive: predicted vs actual over time (does the model track turning points?) and residuals vs time (any obvious heteroskedasticity or regime shifts?).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

axes[0].plot(test.index, y_test, label='Actual', color='black', linewidth=1.5)
axes[0].plot(test.index, yhat_tuned, label='RF tuned', color='crimson', linewidth=1.5)
axes[0].axhline(0, color='gray', linewidth=0.5)
axes[0].set_ylabel('GDP growth (QoQ %)')
axes[0].set_title('Predicted vs actual on the held-out test set')
axes[0].legend()

axes[1].plot(test.index, y_test - yhat_tuned, color='steelblue')
axes[1].axhline(0, color='black', linewidth=0.5)
axes[1].set_ylabel('Residual')
axes[1].set_xlabel('Quarter')
axes[1].set_title('Residuals over time')

plt.tight_layout()
plt.show()

<a id="importance"></a>
## Permutation Importance

Built-in tree importances are fast but biased toward high-cardinality features. **Permutation importance** is the honest version: it shuffles each feature on the test set and measures how much performance drops. We saw the same idea in `03_classification/03`.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf_tuned, X_test, y_test, n_repeats=30, random_state=0, n_jobs=-1,
)
imp_df = pd.DataFrame({
    'feature': x_cols,
    'permutation_importance_mean': perm.importances_mean,
    'permutation_importance_std': perm.importances_std,
    'builtin_importance': rf_tuned.feature_importances_,
}).sort_values('permutation_importance_mean', ascending=False)
imp_df

<a id="compare"></a>
## Compare to OLS Baseline

Refit OLS on the same training data so the comparison is apples-to-apples, then put the numbers next to the RF tuned numbers.

In [ ]:
from src.econometrics import fit_ols
import statsmodels.api as sm

ols = fit_ols(train, y_col=y_col, x_cols=x_cols)
X_test_design = sm.add_constant(test[x_cols], has_constant='add')
yhat_ols = ols.predict(X_test_design).to_numpy()
ols_metrics = regression_metrics(y_test, yhat_ols)

summary = pd.DataFrame({
    'OLS':              ols_metrics,
    'RF (default)':     default_metrics,
    'RF (tuned)':       tuned_metrics,
})
summary

<a id="reflection"></a>
## Reflection

1. Did Random Forest beat OLS on test RMSE? On test R²? If RF won on RMSE but lost on R², what does that imply about the kind of mistakes RF makes?
2. Look at the predicted-vs-actual plot. Does the RF model track GDP turning points (peaks/troughs)? Where does it lag the actual series, and why is that an inherent challenge with tree models in time series?
3. The permutation importance ranking — does it match what economic theory would predict? (e.g., is the term spread, `T10Y2Y_lag1`, the most important feature?) If the ranking is surprising, name one possible reason.